# Fitting API

## `RO_fitting`

The `RO_fitting` function takes the $T$ and $h$ time series together with the RO-type choices and returns the fitted RO parameters.

## Syntax

In [ ]:
par = RO_fitting(T, h, T_option, h_option, noise_option)
par = RO_fitting(T, h, T_option, h_option, noise_option, method_fitting)
par = RO_fitting(T, h, T_option, h_option, noise_option, method_fitting, dt_fitting)

## Description

- `par = RO_fitting(T, h, T_option, h_option, noise_option)` returns fitted RO parameters from the input time series `T` and `h`. The RO type is specified by `T_option`, `h_option`, and `noise_option`. If no fitting method is supplied, the default method for the specified RO type is used.
- `par = RO_fitting(..., method_fitting)` additionally specifies the fitting method.
- `par = RO_fitting(..., method_fitting, dt_fitting)` additionally specifies the temporal interval of the input `T` and `h` time series. When `dt_fitting = 1.0` (default), the input time series are monthly.

## Input arguments

### `T`

Input SST anomaly time series, specified as a vector (`N×1` or `1×N`).

### `h`

Input thermocline-depth anomaly time series, specified as a vector (`N×1` or `1×N`).

### `T_option`

RO equation form for the deterministic $dT/dt$ terms, specified as a MATLAB scalar structure with five fields:

- `R`
- `F1`
- `b_T`
- `c_T`
- `d_T`

Each field has one of the following values:

| Value | Meaning |
|---:|---|
| `0` | Parameter is not considered |
| `1` | Seasonally constant: $X$ |
| `3` | Annual variation: $X + X_a\sin(\omega t+\phi_a)$ |
| `5` | Annual + semiannual variation: $X + X_a\sin(\omega t+\phi_a)+X_{sa}\sin(2\omega t+\phi_{sa})$ |

where $\omega=2\pi/12$ month$^{-1}$.

In [ ]:
T_option = struct( ...
    "R",   X1, ...
    "F1",  X2, ...
    "b_T", X3, ...
    "c_T", X4, ...
    "d_T", X5);

### `h_option`

RO equation form for the deterministic $dh/dt$ terms, specified as a MATLAB scalar structure with three fields:

- `F2`
- `epsilon`
- `b_h`

The field values use the same `0`, `1`, `3`, and `5` convention as `T_option`.

In [ ]:
h_option = struct( ...
    "F2",      X1, ...
    "epsilon", X2, ...
    "b_h",     X3);

### `noise_option`

RO noise options, specified as a MATLAB scalar structure with three fields:

- `T`
- `h`
- `T_type`

The `T` and `h` fields must be identical for `RO_fitting`.

| Field | Value | Meaning |
|---|---|---|
| `T`, `h` | `"white"` | White-noise RO (`n_T = n_h = 1`) |
| `T`, `h` | `"red"` | Red-noise RO (`n_T = n_h = 0`) |
| `T_type` | `"additive"` | Additive-noise RO (`B` not considered; `n_g = 2`) |
| `T_type` | `"multiplicative"` | Multiplicative-noise RO (`n_g = 0`) |
| `T_type` | `"multiplicative-H"` | Multiplicative-Heaviside RO (`n_g = 1`) |

> **Note:** `RO_solver` supports different values of `n_T` and `n_h`, but `RO_fitting` does not because this option is not consistently available for all supported fitting methods.

In [ ]:
noise_option = struct( ...
    "T",      "white", ...
    "h",      "white", ...
    "T_type", "additive");

### `method_fitting`

Fitting method. If omitted, the default method is selected according to the RO type.

| Value | Method |
|---|---|
| `"LR-F"` | Linear regression using a forward-difference tendency |
| `"LR-C"` | Linear regression using a central-difference tendency |
| `"LR-F-MAC"` | Forward-difference linear regression with moment analytical constraints for the `T` noise parameters |
| `"MLE"` | Maximum likelihood estimation using a forward-difference tendency |

### `dt_fitting`

Temporal interval of the input `T` and `h` time series in months (scalar; default = `1.0`).

## Output arguments

### `par`

Fitted RO parameters, returned as a `16×1` cell array:

In [ ]:
par = {R; F1; F2; epsilon; b_T; c_T; d_T; b_h; ...
       sigma_T; sigma_h; B; m_T; m_h; n_T; n_h; n_g};

For `R`, `F1`, `b_T`, `c_T`, and `d_T`:

- `NaN` if the corresponding `T_option` field is `0`
- scalar `X` if the field is `1`
- `[X, Xa, phi_a]` if the field is `3`
- `[X, Xa, phi_a, Xsa, phi_sa]` if the field is `5`

For `F2`, `epsilon`, and `b_h`:

- `NaN` if the corresponding `h_option` field is `0`
- scalar `X` if the field is `1`
- `[X, Xa, phi_a]` if the field is `3`
- `[X, Xa, phi_a, Xsa, phi_sa]` if the field is `5`

For the noise parameters:

- `sigma_T` and `sigma_h` are scalars.
- `B` is `NaN` for additive noise (`n_g = 2`) and scalar for `n_g = 0` or `1`.
- `m_T` and `m_h` are `NaN` for white noise (`n_T = n_h = 1`) and scalar for red noise (`n_T = n_h = 0`).
- `n_T`, `n_h`, and `n_g` correspond to the selected noise options.

## Examples

### Linear-white-additive RO using the default fitting method

The fitted model is

$$
\frac{dT}{dt}=RT+F_1h+b_TT^2+\sigma_Tw_T,
$$

$$
\frac{dh}{dt}=-F_2T-\varepsilon h+\sigma_hw_h.
$$

In [ ]:
T = Tobs;
h = hobs;

T_option = struct( ...
    "R",   1, ...
    "F1",  1, ...
    "b_T", 1, ...
    "c_T", 0, ...
    "d_T", 0);

h_option = struct( ...
    "F2",      1, ...
    "epsilon", 1, ...
    "b_h",     0);

noise_option = struct( ...
    "T",      "white", ...
    "h",      "white", ...
    "T_type", "additive");

par = RO_fitting(T, h, T_option, h_option, noise_option);

### Seasonal-linear-white-multiplicative RO using MLE

The fitted model is

$$
\frac{dT}{dt}
=
\left[R+R_a\sin(\omega_a t+\phi_{Ra})\right]T
+F_1h+b_TT^2+\sigma_T(1+BT)w_T,
$$

$$
\frac{dh}{dt}=-F_2T-\varepsilon h+\sigma_hw_h.
$$

In [ ]:
T = Tobs;
h = hobs;

T_option = struct( ...
    "R",   3, ...
    "F1",  1, ...
    "b_T", 1, ...
    "c_T", 0, ...
    "d_T", 0);

h_option = struct( ...
    "F2",      1, ...
    "epsilon", 1, ...
    "b_h",     0);

noise_option = struct( ...
    "T",      "white", ...
    "h",      "white", ...
    "T_type", "multiplicative");

par = RO_fitting(T, h, T_option, h_option, noise_option, "MLE");

## More about

### Default fitting method by RO type

| RO type | Fitting method |
|---|---|
| Linear-White-Additive | LR-F |
| Linear-White-Multi | MLE |
| Linear-White-Multi-H | MLE |
| Linear-Red-Additive | LR-F |
| Linear-Red-Multi | LR-F-MAC |
| Linear-Red-Multi-H | LR-F |
| Nonlinear-White-Additive | LR-F |
| Nonlinear-White-Multi | LR-F-MAC |
| Nonlinear-White-Multi-H | MLE |
| Nonlinear-Red-Additive | LR-F |
| Nonlinear-Red-Multi | LR-F-MAC |
| Nonlinear-Red-Multi-H | LR-F |
| Seasonal-Linear-White-Additive | LR-F |
| Seasonal-Linear-White-Multi | MLE |
| Seasonal-Linear-White-Multi-H | MLE |
| Seasonal-Linear-Red-Additive | LR-F |
| Seasonal-Linear-Red-Multi | LR-F-MAC |
| Seasonal-Linear-Red-Multi-H | LR-F |
| Seasonal-Nonlinear-White-Additive | LR-F |
| Seasonal-Nonlinear-White-Multi | LR-F-MAC |
| Seasonal-Nonlinear-White-Multi-H | MLE |
| Seasonal-Nonlinear-Red-Additive | LR-F |
| Seasonal-Nonlinear-Red-Multi | LR-F-MAC |
| Seasonal-Nonlinear-Red-Multi-H | LR-F |

The method table is intended to be stored in `RO_fitting_method_table.txt`. If the user does not specify a fitting method, `RO_fitting` loads this table and selects the method according to the RO type.

### Linearization of seasonal parameter fitting

In the master RO equation, seasonal variation in a parameter $X$ is written as

$$
X
+X_a\sin(\omega_a t+\phi_{Xa})
+X_{sa}\sin(2\omega_a t+\phi_{Xsa}).
$$

Directly fitting the five parameters
$X$, $X_a$, $\phi_{Xa}$, $X_{sa}$, and $\phi_{Xsa}$
is a nonlinear fitting problem.

Using trigonometric identities, the same seasonal variation can be written as

$$
X
+X_{as}\sin(\omega_a t)
+X_{ac}\cos(\omega_a t)
+X_{sas}\sin(2\omega_a t)
+X_{sac}\cos(2\omega_a t),
$$

with

$$
X_a=\sqrt{X_{as}^2+X_{ac}^2},
$$

$$
\phi_{Xa}=\operatorname{atan2}\left(\frac{X_{ac}}{X_{as}}\right),
$$

$$
X_{sa}=\sqrt{X_{sas}^2+X_{sac}^2},
$$

$$
\phi_{Xsa}=\operatorname{atan2}\left(\frac{X_{sac}}{X_{sas}}\right).
$$

`RO_fitting` estimates $X$, $X_{as}$, $X_{ac}$, $X_{sas}$, and $X_{sac}$ using the transformed form and then converts them back to the original seasonal amplitude and phase parameters.